In [2]:
import ast
import json
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NOTEBOOK = Path("02_financial_model.ipynb")

if not MODEL_NOTEBOOK.exists():
    MODEL_NOTEBOOK = Path(
        "experiments/03_financial/02_financial_model.ipynb"
    )

with open(MODEL_NOTEBOOK, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Chỉ nạp định nghĩa, không thực thi các cell thử nghiệm
required_classes = {
    "StrictStableKoopmanOperator",
    "SinCosPosEnc",
    "PatchEmbed1D",
    "PatchTST_Backbone",
    "Koopformer_PatchTST",
    "SeriesDecomp",
    "SimpleAutoformer",
    "Koopformer_Autoformer",
    "InformerSparse",
    "Koopformer_Informer",
    "SimpleLSTMForecaster",
}

required_functions = {
    "orthogonalize",
    "compute_loss",
    "create_windows",
    "train_model",
    "evaluate_model",
}

definitions = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if (
            isinstance(node, ast.ClassDef)
            and node.name in required_classes
        ) or (
            isinstance(node, ast.FunctionDef)
            and node.name in required_functions
        ):
            definitions.append(node)

module = ast.Module(body=definitions, type_ignores=[])
ast.fix_missing_locations(module)

exec(
    compile(module, str(MODEL_NOTEBOOK), "exec"),
    globals()
)

missing = [
    name for name in required_classes | required_functions
    if name not in globals()
]

assert not missing, f"Missing definitions: {missing}"

print("Device:", DEVICE)
print("Imported classes:", len(required_classes))
print("Imported functions:", len(required_functions))
print("All model definitions loaded successfully.")

Device: cuda
Imported classes: 11
Imported functions: 5
All model definitions loaded successfully.


In [3]:
DATA_PATH = Path("data/financial_train_test.pt")

if not DATA_PATH.exists():
    DATA_PATH = Path(
        "experiments/03_financial/data/financial_train_test.pt"
    )

bundle = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False
)

train_data = bundle["train"]
test_data = bundle["test"]

INPUT_DIM = 2

# Grid Search
PATCH_LENGTHS = [40, 50, 60, 70, 80, 90, 100]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

MODELS = [
    "LSTM",
    "PatchTST",
    "Autoformer",
    "Informer"
]

# Training hyperparameters
D_MODEL = 48
EPOCHS = 4000
LEARNING_RATE = 3e-4
LYAP_WEIGHT = 0.1

TOTAL_RUNS = (
    len(PATCH_LENGTHS)
    * len(HORIZONS)
    * len(MODELS)
)

print("Train:", train_data.shape)
print("Test:", test_data.shape)
print("Channels:", bundle["channels"])

print("\nNumber of models:", len(MODELS))
print("Patch lengths:", PATCH_LENGTHS)
print("Forecast horizons:", HORIZONS)

print("\nTotal configurations:", TOTAL_RUNS)
print("Epochs per configuration:", EPOCHS)
print("Dataset hash:", bundle["data_hash"])

Train: torch.Size([2320, 2])
Test: torch.Size([580, 2])
Channels: ['BTC', 'ETH']

Number of models: 4
Patch lengths: [40, 50, 60, 70, 80, 90, 100]
Forecast horizons: [2, 4, 6, 8, 10, 12, 14, 16]

Total configurations: 224
Epochs per configuration: 4000
Dataset hash: 47a605f9cf091101afa25f139b28d1cacd406ae4c014b65969a6d4b99ddd49a8


In [4]:
def create_model(model_name, patch_len, horizon, d_model=48):

    if model_name == "LSTM":
        model = SimpleLSTMForecaster(
            input_dim=INPUT_DIM,
            hidden_dim=d_model,
            num_layers=2,
            horizon=horizon
        )

    elif model_name == "PatchTST":
        model = Koopformer_PatchTST(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    elif model_name == "Autoformer":
        model = Koopformer_Autoformer(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    elif model_name == "Informer":
        model = Koopformer_Informer(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model.to(DEVICE)


# Kiểm tra khởi tạo cả 4 kiến trúc
p = 40
H = 16

x_sample = train_data[:p].unsqueeze(0).to(DEVICE)

for model_name in MODELS:

    model = create_model(
        model_name=model_name,
        patch_len=p,
        horizon=H,
        d_model=D_MODEL
    )

    model.eval()

    with torch.no_grad():
        prediction = model(x_sample)

    params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    print(
        f"{model_name:12s} | "
        f"Output: {tuple(prediction.shape)} | "
        f"Parameters: {params:,}"
    )

    assert prediction.shape == (1, H * INPUT_DIM)

    del model

print("\nAll four models initialized successfully.")

LSTM         | Output: (1, 32) | Parameters: 30,368
PatchTST     | Output: (1, 32) | Parameters: 67,040
Autoformer   | Output: (1, 32) | Parameters: 50,864
Informer     | Output: (1, 32) | Parameters: 66,992

All four models initialized successfully.


In [5]:
def run_one_configuration(model_name, patch_len, horizon):

    # 1. Checkpoint directory
    results_dir = (
        DATA_PATH.parent.parent
        / "results"
        / "financial_grid_search_e4000"
    )
    results_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = results_dir / (
        f"{model_name.lower()}_p{patch_len}"
        f"_h{horizon}_d{D_MODEL}.pt"
    )

    # 2. Skip configurations already completed
    if checkpoint_path.exists():

        saved = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        assert saved["data_hash"] == bundle["data_hash"]
        assert saved["epochs"] == EPOCHS
        assert saved["model"] == model_name
        assert saved["patch_len"] == patch_len
        assert saved["horizon"] == horizon
        assert saved["d_model"] == D_MODEL

        print(f"SKIP: {checkpoint_path.name}")

        return checkpoint_path

    print(
        f"\nSTART: {model_name} | "
        f"P={patch_len}, H={horizon}, "
        f"d_model={D_MODEL}"
    )

    # 3. Create sliding windows
    X_train, y_train = create_windows(
        train_data, patch_len, horizon
    )

    X_test, y_test = create_windows(
        test_data, patch_len, horizon
    )

    # 4. Initialize model
    model = create_model(
        model_name=model_name,
        patch_len=patch_len,
        horizon=horizon,
        d_model=D_MODEL
    )

    # 5. Select Lyapunov weight
    lyap_weight = (
        0.0 if model_name == "LSTM"
        else LYAP_WEIGHT
    )

    # 6. Train the model
    result = train_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        lyap_weight=lyap_weight,
        verbose=True
    )

    trained_model = result["model"]

    # 7. Evaluate Train and Test
    train_metrics = evaluate_model(
        trained_model, X_train, y_train
    )

    test_metrics = evaluate_model(
        trained_model, X_test, y_test
    )

    # 8. Build checkpoint
    checkpoint = {
        "model": model_name,

        "model_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in trained_model.state_dict().items()
        },

        "pred_train": train_metrics["predictions"],
        "truth_train": y_train.cpu(),

        "pred_test": test_metrics["predictions"],
        "truth_test": y_test.cpu(),

        "loss": torch.tensor(result["loss"]),
        "eig": torch.tensor(result["eig"]),
        "mse_history": torch.tensor(result["mse"]),
        "lyap_history": torch.tensor(result["lyap"]),

        "train_mse": train_metrics["MSE"],
        "train_mae": train_metrics["MAE"],

        "test_mse": test_metrics["MSE"],
        "test_mae": test_metrics["MAE"],

        "patch_len": patch_len,
        "horizon": horizon,
        "d_model": D_MODEL,
        "input_dim": INPUT_DIM,

        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lyap_weight": lyap_weight,

        "channels": bundle["channels"],
        "data_hash": bundle["data_hash"],
    }

    # 9. Atomic checkpoint save
    temp_path = checkpoint_path.with_suffix(".pt.tmp")

    torch.save(checkpoint, temp_path)
    temp_path.replace(checkpoint_path)

    print(
        f"DONE: {model_name} | "
        f"P={patch_len}, H={horizon}"
    )

    print(
        f"Train MSE: {train_metrics['MSE']:.6f} | "
        f"Test MSE: {test_metrics['MSE']:.6f}"
    )

    print(f"Saved: {checkpoint_path}")

    # 10. Release GPU memory after completed run
    del trained_model, model, result
    del X_train, y_train, X_test, y_test

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return checkpoint_path


print("Financial training function defined successfully.")

Financial training function defined successfully.


In [7]:
from itertools import product
import os

RESULTS_DIR = (
    DATA_PATH.parent.parent
    / "results"
    / "financial_grid_search_e4000"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = RESULTS_DIR / "metrics.csv"

METRIC_COLUMNS = [
    "Model", "PatchLen", "Horizon",
    "d_model", "Set", "Epochs", "MSE", "MAE"
]

CONFIGURATIONS = list(
    product(PATCH_LENGTHS, HORIZONS, MODELS)
)


def checkpoint_file(model_name, patch_len, horizon):
    return RESULTS_DIR / (
        f"{model_name.lower()}_p{patch_len}"
        f"_h{horizon}_d{D_MODEL}.pt"
    )


def collect_metrics():
    """
    Rebuild metrics.csv from completed checkpoints.
    Checkpoints are the source of truth.
    """

    rows = []
    completed = 0

    for patch_len, horizon, model_name in CONFIGURATIONS:

        path = checkpoint_file(
            model_name, patch_len, horizon
        )

        if not path.exists():
            continue

        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        # Verify checkpoint identity
        assert ckpt["data_hash"] == bundle["data_hash"]
        assert ckpt["epochs"] == EPOCHS
        assert ckpt["model"] == model_name
        assert ckpt["patch_len"] == patch_len
        assert ckpt["horizon"] == horizon
        assert ckpt["d_model"] == D_MODEL

        completed += 1

        for dataset in ["Train", "Test"]:

            prefix = dataset.lower()

            rows.append({
                "Model": model_name,
                "PatchLen": patch_len,
                "Horizon": horizon,
                "d_model": D_MODEL,
                "Set": dataset,
                "Epochs": EPOCHS,
                "MSE": ckpt[f"{prefix}_mse"],
                "MAE": ckpt[f"{prefix}_mae"]
            })

    metrics_df = pd.DataFrame(
        rows, columns=METRIC_COLUMNS
    )

    # Atomic CSV save
    temp_path = METRICS_PATH.with_suffix(".csv.tmp")

    metrics_df.to_csv(temp_path, index=False)
    os.replace(temp_path, METRICS_PATH)

    return completed, metrics_df


def run_grid_search(max_new_runs=None):

    completed, _ = collect_metrics()

    print(f"Total configurations: {len(CONFIGURATIONS)}")
    print(f"Already completed: {completed}")
    print(f"Epochs per configuration: {EPOCHS}")
    print(f"Device: {DEVICE}")

    new_runs = 0

    for index, (patch_len, horizon, model_name) in enumerate(
        CONFIGURATIONS, start=1
    ):

        path = checkpoint_file(
            model_name, patch_len, horizon
        )

        if path.exists():
            print(f"SKIP [{index}/224] {path.name}")
            continue

        if max_new_runs is not None and new_runs >= max_new_runs:
            break

        print(
            f"\n[{index}/224] "
            f"{model_name} | P={patch_len}, H={horizon}"
        )

        # Train and save checkpoint
        run_one_configuration(
            model_name=model_name,
            patch_len=patch_len,
            horizon=horizon
        )

        new_runs += 1

        # Update CSV immediately after each completed run
        completed, metrics_df = collect_metrics()

        print(
            f"PROGRESS: {completed}/224 | "
            f"Metrics rows: {len(metrics_df)}"
        )

    completed, metrics_df = collect_metrics()

    print("\nGRID SEARCH STATUS")
    print("Completed:", completed, "/224")
    print("New runs:", new_runs)
    print("Metrics rows:", len(metrics_df))
    print("Results directory:", RESULTS_DIR)

    return metrics_df


print("Grid Search functions defined successfully.")
print("Total configurations:", len(CONFIGURATIONS))

Grid Search functions defined successfully.
Total configurations: 224


In [9]:
import ast
import json
from pathlib import Path

PROJECT_DIR = MODEL_NOTEBOOK.parent.resolve()
TRAIN_NOTEBOOK = PROJECT_DIR / "03_financial_training.ipynb"
OUTPUT_SCRIPT = PROJECT_DIR / "train_financial.py"

model_definitions = [
    "orthogonalize",
    "StrictStableKoopmanOperator",
    "SinCosPosEnc",
    "PatchEmbed1D",
    "PatchTST_Backbone",
    "Koopformer_PatchTST",
    "SeriesDecomp",
    "SimpleAutoformer",
    "Koopformer_Autoformer",
    "InformerSparse",
    "Koopformer_Informer",
    "SimpleLSTMForecaster",
    "create_windows",
    "compute_loss",
    "train_model",
    "evaluate_model",
]

training_definitions = [
    "create_model",
    "run_one_configuration",
    "checkpoint_file",
    "collect_metrics",
    "run_grid_search",
]


def extract_definitions(notebook_path, names):
    with open(notebook_path, encoding="utf-8") as f:
        notebook = json.load(f)

    extracted = {}

    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:
            if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
                if node.name in names:
                    extracted[node.name] = ast.get_source_segment(
                        source, node
                    )

    missing = set(names) - set(extracted)

    if missing:
        raise ValueError(
            f"Missing definitions in {notebook_path}: {missing}"
        )

    return "\n\n\n".join(
        extracted[name] for name in names
    )


model_code = extract_definitions(
    MODEL_NOTEBOOK,
    model_definitions
)

training_code = extract_definitions(
    TRAIN_NOTEBOOK,
    training_definitions
)

header = '''
import argparse
import os
import random
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "data" / "financial_train_test.pt"

bundle = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False
)

train_data = bundle["train"]
test_data = bundle["test"]

INPUT_DIM = 2

PATCH_LENGTHS = [40, 50, 60, 70, 80, 90, 100]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]
MODELS = ["LSTM", "PatchTST", "Autoformer", "Informer"]

D_MODEL = 48
EPOCHS = 4000
LEARNING_RATE = 3e-4
LYAP_WEIGHT = 0.1

RESULTS_DIR = BASE_DIR / "results" / "financial_grid_search_e4000"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = RESULTS_DIR / "metrics.csv"

METRIC_COLUMNS = [
    "Model", "PatchLen", "Horizon",
    "d_model", "Set", "Epochs", "MSE", "MAE"
]

CONFIGURATIONS = list(product(
    PATCH_LENGTHS, HORIZONS, MODELS
))
'''

footer = '''
if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--limit",
        type=int,
        default=1,
        help="Maximum new configurations to train"
    )

    args = parser.parse_args()

    if args.limit < 1:
        parser.error("--limit must be >= 1")

    print("Device:", DEVICE)
    print("GPU:", torch.cuda.get_device_name(0)
          if torch.cuda.is_available() else "CPU")
    print("Total configurations:", len(CONFIGURATIONS))
    print("Epochs:", EPOCHS)
    print("Dataset hash:", bundle["data_hash"])

    run_grid_search(max_new_runs=args.limit)
'''

script = (
    header
    + "\n\n"
    + model_code
    + "\n\n"
    + training_code
    + "\n\n"
    + footer
)

# Validate Python syntax without starting training
compile(script, str(OUTPUT_SCRIPT), "exec")

OUTPUT_SCRIPT.write_text(script, encoding="utf-8")

print("Script created:", OUTPUT_SCRIPT)
print("File size:", round(OUTPUT_SCRIPT.stat().st_size / 1024, 2), "KB")
print("Model definitions:", len(model_definitions))
print("Training definitions:", len(training_definitions))
print("Syntax check: PASSED")
print("Ready for GPU deployment.")

Script created: C:\Users\Win 11 New\Desktop\pa gk\experiments\03_financial\train_financial.py
File size: 26.17 KB
Model definitions: 16
Training definitions: 5
Syntax check: PASSED
Ready for GPU deployment.
